In [0]:
from pyspark.sql.functions import col, when, explode, map_values
from datetime import date
from dateutil.relativedelta import relativedelta
from delta.tables import *
from time import time

In [ ]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# Get data from 'runescape.01_bronze.latest_prices_raw' remove duplicates
df_latest_prices = spark.read.table(f"{catalog}.01_bronze.latest_prices_raw").dropDuplicates()
# filter data to only last 15 mintutes and remove prices for items we will never be trading
# job will run this notebook every 10 minutes
unix_timestamp = int(time())
df_latest_prices = df_latest_prices.filter(f"(time > {unix_timestamp} - 900) and (price < 35000)")

In [0]:
# Write cleansed data to a Unity Catalog managed Delta table in the silver schema
df_latest_prices.write.mode("overwrite").saveAsTable(f"{catalog}.02_silver.latest_prices_cleansed")